In [ ]:
rundir = '/gpfs/commons/home/tbotella/bam-readcount-rs/bench/results/latest'

In [ ]:
from pathlib import Path
import polars as pl
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'svg'
mpl.rcParams['figure.dpi'] = 110
mpl.rcParams['savefig.dpi'] = 200
mpl.rcParams['savefig.bbox'] = 'tight'
mpl.rcParams['font.family'] = 'DejaVu Sans'
rundir = Path(rundir)
plots = rundir / 'plots'
plots.mkdir(parents=True, exist_ok=True)
print('rundir:', rundir)
print('plots:', plots)

In [ ]:
INFO_COLS = [
    'count',
    'avg_mapping_quality',
    'avg_basequality',
    'avg_se_mapping_quality',
    'num_plus_strand',
    'num_minus_strand',
    'avg_pos_as_fraction',
    'avg_num_mismatches_as_fraction',
    'avg_sum_mismatch_qualities',
    'num_q2_containing_reads',
    'avg_distance_to_q2_start_in_q2_reads',
    'avg_clipped_length',
    'avg_distance_to_effective_3p_end',
]

def parse_brc(path):
    """Parse a bam-readcount output into a long polars frame: one row per (chr,pos,base).
    Returns columns: chrom, pos, ref, depth, base, count, <13 metrics>."""
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 5:
                continue
            chrom, pos, ref, depth = parts[0], int(parts[1]), parts[2], int(parts[3])
            for field in parts[4:]:
                vals = field.split(':')
                base = vals[0]
                if base == '=':
                    continue
                if int(vals[1]) == 0:
                    continue
                row = {'chrom': chrom, 'pos': pos, 'ref': ref, 'depth': depth, 'base': base}
                for k, v in zip(INFO_COLS, vals[1:]):
                    row[k] = float(v)
                rows.append(row)
    if not rows:
        return pl.DataFrame({c: [] for c in ['chrom','pos','ref','depth','base'] + INFO_COLS})
    return pl.DataFrame(rows)


In [ ]:
raw_dirs = sorted([p for p in (rundir / 'raw').iterdir() if p.is_dir()])
print(f'Found {len(raw_dirs)} sample dirs')

frames = []
skipped = []
for d in raw_dirs:
    rs = d / 'rs.txt'
    rf = d / 'ref.txt'
    if not rs.exists() or not rf.exists() or rs.stat().st_size == 0 or Path(rf).resolve().stat().st_size == 0:
        skipped.append(d.name)
        continue
    try:
        df_rs = parse_brc(rs).with_columns(pl.lit(d.name).alias('sample_id'), pl.lit('rs').alias('tool'))
        df_ref = parse_brc(rf).with_columns(pl.lit(d.name).alias('sample_id'), pl.lit('ref').alias('tool'))
    except Exception as e:
        print('parse failed:', d.name, e)
        skipped.append(d.name)
        continue
    if len(df_rs) == 0 or len(df_ref) == 0:
        skipped.append(d.name)
        continue
    frames.append((d.name, df_rs, df_ref))
print(f'Parsed {len(frames)} samples, skipped {len(skipped)}')
if skipped:
    print('skipped:', skipped[:10])

In [ ]:
# Build a single joined long frame: one row per (sample,chrom,pos,base) with paired ref_<m> and rs_<m> columns.
joined_parts = []
for sid, df_rs, df_ref in frames:
    rs2 = df_rs.rename({c: f'rs_{c}' for c in INFO_COLS})
    rf2 = df_ref.rename({c: f'ref_{c}' for c in INFO_COLS}).drop(['ref','depth'])
    j = rs2.join(rf2, on=['sample_id','chrom','pos','base'], how='inner', suffix='_ref')
    joined_parts.append(j)
joined = pl.concat(joined_parts) if joined_parts else None
print('joined rows:', 0 if joined is None else len(joined))
if joined is not None:
    print(joined.head())


In [ ]:
from scipy import stats
rows = []
for m in INFO_COLS:
    a = joined[f'ref_{m}'].to_numpy()
    b = joined[f'rs_{m}'].to_numpy()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    n = len(a)
    if n == 0:
        rows.append({'metric': m, 'n': 0, 'pearson_r': float('nan'), 'spearman_rho': float('nan'),
                     'mae': float('nan'), 'pct_within_0.01': float('nan'),
                     'pct_exact': float('nan')})
        continue
    pr = stats.pearsonr(a, b).statistic if a.std() > 0 and b.std() > 0 else 1.0
    sr = stats.spearmanr(a, b).statistic if a.std() > 0 and b.std() > 0 else 1.0
    mae = float(np.mean(np.abs(a - b)))
    pct_close = float(np.mean(np.abs(a - b) <= 0.01) * 100.0)
    pct_exact = float(np.mean(a == b) * 100.0)
    rows.append({'metric': m, 'n': n, 'pearson_r': pr, 'spearman_rho': sr,
                 'mae': mae, 'pct_within_0.01': pct_close, 'pct_exact': pct_exact})
corr_df = pl.DataFrame(rows)
corr_df.write_csv(rundir / 'per_feature_corr.tsv', separator='\t')
print(corr_df)


In [ ]:
# CORRELATION GRID
import math
n_metrics = len(INFO_COLS)
ncols = 4
nrows = math.ceil(n_metrics / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.0*ncols, 3.6*nrows))
axes = axes.flatten()
panel_letters = list('ABCDEFGHIJKLMN')
for i, m in enumerate(INFO_COLS):
    ax = axes[i]
    a = joined[f'ref_{m}'].to_numpy()
    b = joined[f'rs_{m}'].to_numpy()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) > 50000:
        idx = np.random.default_rng(0).choice(len(a), size=50000, replace=False)
        a, b = a[idx], b[idx]
    r = corr_df.filter(pl.col('metric') == m)['pearson_r'][0]
    pct = corr_df.filter(pl.col('metric') == m)['pct_exact'][0]
    ax.scatter(a, b, s=1.5, alpha=0.18, edgecolor='none', color='#3367d6')
    lo = min(a.min(), b.min()) if len(a) > 0 else 0
    hi = max(a.max(), b.max()) if len(a) > 0 else 1
    ax.plot([lo, hi], [lo, hi], color='#bbbbbb', lw=0.8, ls='--', zorder=0)
    ax.set_xlabel(f'upstream {m}', fontsize=8)
    ax.set_ylabel(f'rs {m}', fontsize=8)
    ax.tick_params(labelsize=7)
    color = '#0a8a3a' if r >= 0.99 else ('#c4a000' if r >= 0.95 else '#a30000')
    ax.set_title(f'{panel_letters[i]}  {m}\nr={r:.4f}  exact={pct:.1f}%', fontsize=8.5, color=color)
for j in range(n_metrics, len(axes)):
    axes[j].axis('off')
fig.suptitle(f'bam-readcount-rs vs upstream — {len(joined):,} (sample,position,base) rows across {len(frames)} samples', fontsize=11, y=1.005)
fig.tight_layout()
fig.savefig(plots / 'correlation_grid.png')
plt.show()


In [ ]:
# RUNTIME (single-thread bam-readcount-rs only — upstream timing comes from
# Nextflow trace files in the STREGA pipeline, see README for details.)
ms = pl.read_csv(rundir / 'per_sample_metrics.tsv', separator='\t')
ms = ms.with_columns([
    pl.col('rs_wall_s').cast(pl.Float64),
    pl.col('rs_peak_rss_kb').cast(pl.Float64),
    pl.col('bed_lines').cast(pl.Int64),
])
ms = ms.sort('bed_lines')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
for c in ms['cohort'].unique().to_list():
    sub = ms.filter(pl.col('cohort') == c)
    ax.scatter(sub['bed_lines'], sub['rs_wall_s'], label=c, s=14, alpha=0.7)
ax.set_xlabel('queried sites (BED lines)')
ax.set_ylabel('bam-readcount-rs wall time (s, single thread)')
ax.set_title(f'Runtime — median {ms["rs_wall_s"].median():.1f}s, n={len(ms)}')
ax.legend(fontsize=8, frameon=False)
ax.set_xscale('log')
ax = axes[1]
for c in ms['cohort'].unique().to_list():
    sub = ms.filter(pl.col('cohort') == c)
    ax.scatter(sub['bed_lines'], sub['rs_peak_rss_kb'] / 1024.0, label=c, s=14, alpha=0.7)
ax.set_xlabel('queried sites (BED lines)')
ax.set_ylabel('bam-readcount-rs peak RSS (MB, single thread)')
ax.set_title(f'Memory — median {ms["rs_peak_rss_kb"].median()/1024:.1f}MB, n={len(ms)}')
ax.legend(fontsize=8, frameon=False)
ax.set_xscale('log')
fig.tight_layout()
fig.savefig(plots / 'runtime_memory.png')
plt.show()


In [ ]:
# SUMMARY
lines = [
    f'# Benchmark summary — `{rundir.name}`',
    '',
    f'- samples: **{len(frames)}** (skipped {len(skipped)})',
    f'- joined rows: **{len(joined):,}** (per-sample × position × base)',
    f'- median rs wall time: **{ms["rs_wall_s"].median():.1f}s** (single thread)',
    f'- median rs peak RSS: **{ms["rs_peak_rss_kb"].median()/1024:.1f} MB**',
    '',
    '## Per-feature correlation',
    '',
    '| metric | n | pearson r | spearman ρ | MAE | % exact | % within 0.01 |',
    '|---|---:|---:|---:|---:|---:|---:|',
]
for r in corr_df.iter_rows(named=True):
    lines.append(f"| {r['metric']} | {r['n']:,} | {r['pearson_r']:.5f} | {r['spearman_rho']:.5f} | {r['mae']:.4g} | {r['pct_exact']:.2f}% | {r['pct_within_0.01']:.2f}% |")
(rundir / 'SUMMARY.md').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
